# Virtual-pipeline — Commercial VTuber Builder

이미지 생성은 이 노트북 밖에서 합니다. 입력 캐릭터/악세사리 이미지는 단순한 중간 회색 배경, 한 객체, 가림 최소화를 권장합니다.

- Avatar Mode: 이미지 → TripoSR → canonical fitting → rig/expressions/gaze/SpringBone → VRM 1.0
- Accessory Mode: 완성 VRM + 악세사리 이미지들 → 3D → anchor fitting/collision → static bake

필수 단계가 실패하면 파이프라인은 complete를 반환하지 않습니다.

## 1. Setup

In [ ]:
import os, pathlib, subprocess, sys

if sys.version_info < (3, 12):
    raise RuntimeError(f'Python 3.12+ required, current={sys.version}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'], check=True)

triposr = pathlib.Path('/content/third_party/TripoSR')
triposr.parent.mkdir(parents=True, exist_ok=True)
if not triposr.exists():
    subprocess.run(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR.git', str(triposr)], check=True)
subprocess.run(['git', '-C', str(triposr), 'checkout', '107cefdc244c39106fa830359024f6a2f1c78871'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(triposr / 'requirements.txt')], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-U',
    'git+https://github.com/jujumelona/Virtual-pipeline.git'
], check=True)

os.environ['TRIPOSR_DIR'] = str(triposr)
print('Setup complete:', os.environ['TRIPOSR_DIR'])

## 2. Avatar Mode

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Avatar Mode에서는 캐릭터 이미지 1장만 업로드하세요.')

name, data = next(iter(uploaded.items()))
avatar_input_dir = Path('/content/input/avatar')
avatar_input_dir.mkdir(parents=True, exist_ok=True)
avatar_image = avatar_input_dir / name
avatar_image.write_bytes(data)
print('Avatar input:', avatar_image)

In [ ]:
import json
from vtuber_pipeline.avatar import build_avatar

avatar_output = '/content/output/avatar'
avatar_result = build_avatar(
    image_path=str(avatar_image),
    output_dir=avatar_output,
    config={
        'profile': 'commercial',
        'commercial_usage': 'corporation',
    },
)

print(json.dumps({
    'status': avatar_result.get('status'),
    'failed_stages': avatar_result.get('failed_stages'),
    'failed_reason': avatar_result.get('failed_reason'),
    'vrm_path': avatar_result.get('vrm_path'),
}, ensure_ascii=False, indent=2))

if avatar_result.get('status') != 'complete':
    for stage, result in avatar_result.get('stages', {}).items():
        print(stage, '=>', result.get('status'), result.get('error') or result.get('warning'))
    raise RuntimeError('Avatar build did not complete.')

avatar_vrm_path = avatar_result['vrm_path']
print('Avatar VRM:', avatar_vrm_path)

In [ ]:
from google.colab import files
files.download(avatar_vrm_path)

## 3. Accessory Mode

Avatar Mode가 끝난 뒤 별도로 실행합니다. 여러 이미지를 한 번에 업로드할 수 있습니다.

ANCHORS를 업로드 순서에 맞게 수정할 수 있습니다.
지원 예: HEAD_TOP, FACE, LEFT_EAR, RIGHT_EAR, NECK, CHEST, BACK, LEFT_HAND, RIGHT_HAND, HIPS.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded_accessories = files.upload()
acc_input_dir = Path('/content/input/accessories')
acc_input_dir.mkdir(parents=True, exist_ok=True)

accessory_images = []
for name, data in uploaded_accessories.items():
    p = acc_input_dir / name
    p.write_bytes(data)
    accessory_images.append(str(p))

ANCHORS = ['HEAD_TOP'] * len(accessory_images)
print(list(zip(accessory_images, ANCHORS)))

In [ ]:
from vtuber_pipeline.accessory import reconstruct_accessories, AccessoryPipeline

recon_dir = '/content/output/accessory_reconstruction'
reconstructed = reconstruct_accessories(accessory_images, recon_dir)

current_vrm = avatar_vrm_path
for idx, (item, anchor) in enumerate(zip(reconstructed, ANCHORS)):
    if item.get('status') != 'complete':
        raise RuntimeError(f'Accessory reconstruction failed: {item}')

    out_dir = f'/content/output/accessory_{idx:03d}'
    result = AccessoryPipeline(out_dir).build(
        base_vrm=current_vrm,
        accessory_glb=item['mesh'],
        config={
            'anchor_name': anchor,
            'physics': {'enabled': False},
        },
    )
    print(idx, anchor, result.get('status'))
    if result.get('status') != 'complete':
        raise RuntimeError(f'Accessory pipeline failed: {result}')
    current_vrm = result['output_vrm']

combined_vrm_path = current_vrm
print('Combined VRM:', combined_vrm_path)

In [ ]:
from google.colab import files
files.download(combined_vrm_path)